# Day 4 — Clean Continuous Pipeline (No Bronze)

**Goal:** Predict flood risk using a clean Medallion architecture where raw data is purged after processing.

**Architecture:** 
1. **Ingestion**: Live forecast is processed in-memory.
2. **Persistence**: Only standardized Silver and engineered Gold layers are saved to disk.
3. **Context**: 6-hour resampled stream ensures continuity without storing raw bytes.

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add src to path
sys.path.append(os.path.abspath('../'))
from src import config, ingestion, pipeline

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

## 1. Fetch Live Forecast Data

In [ ]:
# Fetch 15-day forecast for all Baku zones
forecast_frames = []
for zone in config.BAKU_ZONES:
    df_z = ingestion.fetch_weather_forecast(zone['zone'], zone['latitude'], zone['longitude'], forecast_days=15)
    forecast_frames.append(df_z)

df_forecast_raw = pd.concat(forecast_frames)
print(f"Live forecast data fetched: {len(df_forecast_raw)} hourly rows.")

## 2. Execute Clean Pipeline
This step resamples data to Silver and purges raw Bronze data from disk.

In [ ]:
# Process live data through the clean pipeline
# Note: Bronze tables are not persisted in this mode.
df_features = pipeline.run_forecast_pipeline(df_forecast_raw)

print(f"Pipeline complete. {len(df_features)} rows generated for prediction window.")
df_features.head()

## 3. Predict & Alert

In [ ]:
checkpoint = joblib.load('../models/baku_sentinel_rf.joblib')
model = checkpoint['model']
feature_cols = checkpoint['features']

# Prepare and Sync features
X_live = pd.get_dummies(df_features.drop(columns=['time_6h', 'river_discharge', 'is_flood']), columns=['zone'], drop_first=True)
for col in feature_cols: 
    if col not in X_live.columns: X_live[col] = 0
X_live = X_live[feature_cols].fillna(0)

# Calculate Inundation Probability
df_features['risk_score'] = model.predict_proba(X_live)[:, 1]

# Summary Alert
display(df_features.groupby('zone').agg({'risk_score': ['max', 'mean']}))